In [ ]:
import os
import sys
from pathlib import Path

import boto3
from sagemaker.core import image_uris
from sagemaker.core.helper.session_helper import Session
from sagemaker.core.training.configs import Compute, InputData, SourceCode
from sagemaker.train.model_trainer import Mode, ModelTrainer

NOTEBOOK_PATH = Path(globals().get('__vsc_ipynb_file__', Path.cwd() / 'train_local.ipynb')).resolve()
TRAINING_DIR = NOTEBOOK_PATH.parent

from src.train.static_embedding.shared import (
    build_job_name,
    derive_job_basename,
    find_project_root,
    set_global_seed,
    prepare_dataset_files,
)

PROJECT_ROOT = find_project_root(TRAINING_DIR)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.lib.sagemaker import CODE_STAGE_EXCLUDE, patch_sagemaker
from src.preprocess.standardize.two_car_pros import DATASET_LANCEDB_DIR, LANCEDB_STANDARDIZED_RECORDS

patch_sagemaker()
SEED = int(os.getenv('SEED', '42'))
set_global_seed(SEED)

region = boto3.Session().region_name or os.getenv('AWS_DEFAULT_REGION', 'us-east-1')
role = os.getenv('SAGEMAKER_EXECUTION_ROLE_ARN', 'arn:aws:iam::000000000000:role/SageMakerRole')
sess = Session()

JOB_BASENAME = derive_job_basename(TRAINING_DIR)
JOB_NAME = build_job_name(TRAINING_DIR)
LOCAL_WORK_DIR = TRAINING_DIR / 'artifacts' / JOB_BASENAME
LOCAL_DATA_DIR = LOCAL_WORK_DIR / 'input'
RUN_RUNTIME_ROOT = LOCAL_WORK_DIR / 'local_runtime_runs'
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
RUN_RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)

LOCAL_INSTANCE_TYPE = 'local_gpu'

print(f'Project root: {PROJECT_ROOT}')
print(f'Training dir: {TRAINING_DIR}')
print(f'Job basename: {JOB_BASENAME}')
print(f'Example job name: {JOB_NAME}')
print(f'Region: {region}')
print(f'Local instance type: {LOCAL_INSTANCE_TYPE}')

In [ ]:
train_pairs, eval_pairs, train_file, eval_file = prepare_dataset_files(
    db_dir=DATASET_LANCEDB_DIR,
    table_name=LANCEDB_STANDARDIZED_RECORDS,
    output_dir=LOCAL_DATA_DIR,
    seed=SEED,
    eval_ratio=0.2,
    #max_pairs=10000,
)

train_channel_dir = train_file.parent
eval_channel_dir = eval_file.parent

print(f'Train file: {train_file}')
print(f'Eval file: {eval_file}')
print(f'Train rows: {len(train_pairs):,} | Eval rows: {len(eval_pairs):,}')

In [ ]:
from src.train.static_embedding.sagemaker.shared import HyperparametersModel
from src.train.static_embedding.shared import build_mlflow_environment


hf_training_image = image_uris.retrieve(
    framework='huggingface',
    region=region,
    version='4.56.2',
    base_framework_version='pytorch2.8.0',
    py_version='py312',
    image_scope='training',
)

source_code = SourceCode(
    source_dir=str(TRAINING_DIR / 'sagemaker'),
    entry_script='launch.sh',
    ignore_patterns=list(CODE_STAGE_EXCLUDE),
)

train_input = InputData(
    channel_name='train',
    data_source=str(train_channel_dir),
    content_type='application/jsonlines',
)
eval_input = InputData(
    channel_name='eval',
    data_source=str(eval_channel_dir),
    content_type='application/jsonlines',
)

hyperparameters = HyperparametersModel(
    num_train_epochs=5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    seed=SEED,
).model_dump_json()

environment = build_mlflow_environment(
    mode='local',
    experiment_name=JOB_BASENAME,
    tracking_uri=os.getenv('MLFLOW_TRACKING_URI'),
)
print(f"MLflow tracking URI: {environment['MLFLOW_TRACKING_URI']}")

trainer = ModelTrainer(
    training_mode=Mode.LOCAL_CONTAINER,
    training_image=hf_training_image,
    role=role,
    sagemaker_session=sess,
    source_code=source_code,
    local_container_root=str(RUN_RUNTIME_ROOT),
    compute=Compute(instance_count=1, instance_type=LOCAL_INSTANCE_TYPE),
    hyperparameters={'json': hyperparameters},
    base_job_name=JOB_BASENAME,
    environment=environment,
)

trainer.train(
    input_data_config=[train_input, eval_input],
    wait=True,
    logs=False,
)